# __Futures/Spot HBT Strategy Interface Example__

This notebook follows the same staged shape as `hbt_pair_backtest_visualization.ipynb`, but imports the project-level strategy contract from `scripts.strategy_api` and the futures/spot implementation adapter from `future_spot.arbitrage.strategy_adapter`.

In [ ]:
from argparse import Namespace
from pathlib import Path
import sys

import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)

NOTEBOOK_DIR = Path.cwd().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent
FUTURE_SPOT_ROOT = PROJECT_ROOT / "future_spot"

for path in (PROJECT_ROOT, FUTURE_SPOT_ROOT):
    text = str(path)
    if text not in sys.path:
        sys.path.insert(0, text)

from scripts.strategy_api import StrategyContext, StrategyDecision
from future_spot.arbitrage.strategy_adapter import FutureSpotPairStrategy
from future_spot.arbitrage import daily_pipeline, event_data, hbt_pipeline, reporting


## __Parameters__

In [ ]:
args = Namespace(
    start_date="2026-05-21",
    end_date="2026-05-26",
    base_config=FUTURE_SPOT_ROOT / "arbitrage_config_base.json",
    calendar=FUTURE_SPOT_ROOT / "Calendar.csv",
    stockinfo=FUTURE_SPOT_ROOT / "stockinfo.csv",
    output_dir=FUTURE_SPOT_ROOT / "output" / "hbt_strategy_interface_example",
    futures_parquet_template=daily_pipeline.DEFAULT_FUTURES_PARQUET_TEMPLATE,
    twse_daytrade_template=daily_pipeline.DEFAULT_TWSE_DAYTRADE_TEMPLATE,
    tpex_daytrade_template=daily_pipeline.DEFAULT_TPEX_DAYTRADE_TEMPLATE,
    twse_daily_template=daily_pipeline.DEFAULT_TWSE_DAILY_TEMPLATE,
    tpex_daily_template=daily_pipeline.DEFAULT_TPEX_DAILY_TEMPLATE,
    build_session_start="08:45:00",
    build_session_end="13:45:00",
    min_future_volume=1000,
    min_stock_volume=20_000_000,
    required_unit=2000,
    name_template="{spot_symbol}_{future_symbol}",
    rebuild_daily_configs=False,
    session_start="09:00:00",
    session_end="13:30:00",
    pair_name=[],
    max_pairs=1,
    no_convert_missing_event_data=True,
    rebuild_event_data=False,
    conversion_qa_sample_rows=1000,
    spot_input_csv_template=event_data.DEFAULT_SPOT_INPUT_CSV_TEMPLATE,
    data_platform_base="/mnt/z/數據平台",
    event_futures_parquet_dir=None,
    first_leg="future",
    step_ms=1000.0,
    order_latency_ms=0.0,
    response_latency_ms=0.0,
    feed_latency_offset_ms=0.0,
    second_leg_delay_ms=0.0,
    post_first_feed_wait="none",
    post_first_feed_timeout_ms=0.0,
    post_first_feed_poll_ms=1.0,
    response_timeout_ms=50.0,
    max_steps=None,
    max_trades_per_pair=None,
    record_market_every_steps=1,
    rebuild_hbt_results=False,
    queue_model="risk_adverse",
    entry_threshold_pct=None,
    exit_threshold_pct=None,
    min_effective_tick_multiple=None,
    min_second_leg_adjusted_basis_pct=None,
    no_second_leg_profit_check=False,
    no_flatten=False,
    continue_on_error=True,
)
args.output_dir.mkdir(parents=True, exist_ok=True)
strategy = FutureSpotPairStrategy()


## __Build Daily Pair Config Universe__

In [ ]:
trade_dates = daily_pipeline.select_trade_dates(args.calendar, args.start_date, args.end_date)
records, build_status = daily_pipeline.build_daily_pair_records(args, trade_dates)
reporting.write_csv(build_status, args.output_dir / "daily_config_build_status.csv")

pair_universe = daily_pipeline.pair_universe_frame(records)
reporting.write_csv(pair_universe, args.output_dir / "daily_pair_universe.csv")
pair_universe.head(10)


## __Build / Reuse HBT Event Data__

In [ ]:
event_paths, conversion_status = event_data.build_event_data(args, records)
reporting.write_csv(conversion_status, args.output_dir / "conversion_status.csv")
conversion_status.head(10)


## __Run HBT With Strategy Adapter__

In [ ]:
# hbt_pipeline.run_backtests uses the default FutureSpotPairStrategy internally.
# For a custom strategy, instantiate HbtPairBacktester directly with strategy=<your adapter>.
pair_results, summary, trades, market, latency, run_errors = hbt_pipeline.run_or_load_backtests(args, records, event_paths)
reporting.write_csv(summary, args.output_dir / "summary_all_daily_pairs.csv")
reporting.write_csv(trades, args.output_dir / "trades_all_daily_pairs.csv")
reporting.write_csv(market, args.output_dir / "market_all_daily_pairs.csv")
reporting.write_csv(latency, args.output_dir / "latency_all_daily_pairs.csv")
reporting.write_csv(run_errors, args.output_dir / "run_errors.csv")
summary.head(10)


## __Entry / Exit Output__

In [ ]:
entry_exit_by_pair, entry_exit_all, entry_exit_index = reporting.build_entry_exit_outputs(pair_results, records)
reporting.write_csv(entry_exit_all, args.output_dir / "entry_exit_all_daily_pairs.csv")
reporting.write_csv(entry_exit_index, args.output_dir / "entry_exit_index.csv")
entry_exit_index.head(10)
